In [ ]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Function to calculate statistics with confidence intervals
def calculate_stats_with_ci(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    count = len(stat_series)
    ci_lower = mean - 1.96 * (std_dev / (count ** 0.5))
    ci_upper = mean + 1.96 * (std_dev / (count ** 0.5))
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv, ci_lower, ci_upper

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

# Process team rosters
def process_team_roster(team_name):
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            normalized_positions = normalize_player_position(player_position)
            print(f"Processed {player_name} with positions {normalized_positions}")

    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Main analysis for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        return

    for game in games_today:
        print(f"Processing game: {game['home_team']} vs. {game['away_team']}")
        for team_name in [game['home_team'], game['away_team']]:
            process_team_roster(team_name)

    # Adjust for multi-position players in DvP adjustment
    for pos in normalized_positions:
        dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics['TEAM_ABBREVIATION'], 1)
        # Proceed with calculations or aggregations

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Analyze teams and players for today's games
# Main analysis function
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    # Ensure games_today has been fetched
    games_today = fetch_schedule()

    if not games_today:
        print("No games scheduled for today.")
    else:
        all_players_data = []  # You need to define this to collect player data
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']

            for team_name in [home_team, away_team]:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    # Process player data as needed
                    # Your processing logic goes here

            for game in games_today:
                home_team = game['home_team']
                away_team = game['away_team']

                for team_name in [home_team, away_team]:
                    team_id = teams.find_team_by_abbreviation(team_name)['id']
                    roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                    for _, player in roster.iterrows():
                        player_position = player['POSITION']
                        # Continue processing

                    if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                    for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = calculate_stats_with_ci(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg, confidence_pct_rpg, ci_lower_rpg, ci_upper_rpg = calculate_stats_with_ci(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg, confidence_pct_apg, ci_lower_apg, ci_upper_apg = calculate_stats_with_ci(game_logs['AST'])

                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                    # Add new difference columns
                    ppg_diff = mean_ppg - std_dev_ppg
                    rpg_diff = mean_rpg - std_dev_rpg
                    apg_diff = mean_apg - std_dev_apg

                    all_players_data.append({
                        'Player': player_name,
                        'Team': team_name,
                        'PPG': round(mean_ppg, 2),
                        'PPG SD': round(std_dev_ppg, 2),
                        'PPG CI Lower': round(ci_lower_ppg, 2),
                        'PPG CI Upper': round(ci_upper_ppg, 2),
                        'Confidence % PPG': round(confidence_pct_ppg, 2),
                        'PPG CV': round(cv_ppg, 2),
                        'PPG Diff': round(ppg_diff, 2),  # Added PPG Diff
                        'Adj PPG': round(adjusted_ppg, 2),
                        'RPG': round(mean_rpg, 2),
                        'RPG SD': round(std_dev_rpg, 2),
                        'RPG CI Lower': round(ci_lower_rpg, 2),
                        'RPG CI Upper': round(ci_upper_rpg, 2),
                        'Confidence % RPG': round(confidence_pct_rpg, 2),
                        'RPG CV': round(cv_rpg, 2),
                        'RPG Diff': round(rpg_diff, 2),  # Added RPG Diff
                        'Adj RPG': round(adjusted_rpg, 2),
                        'APG': round(mean_apg, 2),
                        'APG SD': round(std_dev_apg, 2),
                        'APG CI Lower': round(ci_lower_apg, 2),
                        'APG CI Upper': round(ci_upper_apg, 2),
                        'Confidence % APG': round(confidence_pct_apg, 2),
                        'APG CV': round(cv_apg, 2),
                        'APG Diff': round(apg_diff, 2),  # Added APG Diff
                        'Adj APG': round(adjusted_apg, 2),
                        'USG%': usg_pct,
                        'Injury Status': injury_status
                    })

                    time.sleep(1)  # To avoid rate-limiting
            except Exception as e:
              print(f"Error analyzing team {team_name}: {e}")
              continue

    df = pd.DataFrame(all_players_data)
    display(df)
    return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()

print(f"Analyzing team: {team_name}")


In [ ]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

# Process team rosters
def process_team_roster(team_name):
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            normalized_positions = normalize_player_position(player_position)
            print(f"Processed {player_name} with positions {normalized_positions}")

    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Main analysis for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        return

    for game in games_today:
        print(f"Processing game: {game['home_team']} vs. {game['away_team']}")
        for team_name in [game['home_team'], game['away_team']]:
            process_team_roster(team_name)

    # Adjust for multi-position players in DvP adjustment
    for pos in normalized_positions:
        dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics['TEAM_ABBREVIATION'], 1)
        # Proceed with calculations or aggregations

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Analyze teams and players for today's games
# Main analysis function
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    # Ensure games_today has been fetched
    games_today = fetch_schedule()

    if not games_today:
        print("No games scheduled for today.")
    else:
        all_players_data = []  # You need to define this to collect player data
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']

            for team_name in [home_team, away_team]:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    # Process player data as needed
                    # Your processing logic goes here

            for game in games_today:
                home_team = game['home_team']
                away_team = game['away_team']

                for team_name in [home_team, away_team]:
                    team_id = teams.find_team_by_abbreviation(team_name)['id']
                    roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                    for _, player in roster.iterrows():
                        player_position = player['POSITION']
                        # Continue processing

                    if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                    for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = calculate_stats_with_ci(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg, confidence_pct_rpg, ci_lower_rpg, ci_upper_rpg = calculate_stats_with_ci(game_logs['REB'])


                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                ppg_diff = mean_ppg - std_dev_ppg
                                rpg_diff = mean_rpg - std_dev_rpg
                                apg_diff = mean_apg - std_dev_apg

                                all_players_data.append({
                                    'Player': player_name,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': round(ppg_diff, 2),  # Added PPG Diff
                                    'Conf % PPG': round(confidence_pct_ppg, 2),
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': round(rpg_diff, 2),  # Added RPG Diff
                                    'Conf % RPG': round(confidence_pct_rpg, 2),
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': round(apg_diff, 2),  # Added APG Diff
                                    'Adj APG': round(adjusted_apg, 2),
                                    'Conf % APG': round(confidence_pct_apg, 2),
                                    'USG%': usg_pct,
                                    'Inj St': injury_status
                                })

                                time.sleep(1)  # To avoid rate-limiting
                        except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                            continue

        df = pd.DataFrame(all_players_data)
        display(df)
        return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


Error fetching player usage metrics: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Error analyzing team MIL: not enough values to unpack (expected 6, got 5)
Error analyzing team ORL: not enough values to unpack (expected 6, got 5)
Error analyzing team MIL: not enough values to unpack (expected 6, got 5)
Error analyzing team ORL: not enough values to unpack (expected 6, got 5)
Error analyzing team OKC: not enough values to unpack (expected 6, got 5)
Error analyzing team DAL: not enough values to unpack (expected 6, got 5)
Error analyzing team OKC: not enough values to unpack (expected 6, got 5)
Error analyzing team DAL: not enough values to unpack (expected 6, got 5)
Error analyzing team MIL: not enough values to unpack (expected 6, got 5)
Error analyzing team ORL: not enough values to unpack (expected 6, got 5)


KeyboardInterrupt: 

In [ ]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# Function to calculate statistics with confidence intervals and Confidence %
def calculate_stats_with_ci(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    count = len(stat_series)
    ci_lower = mean - 1.96 * (std_dev / (count ** 0.5))
    ci_upper = mean + 1.96 * (std_dev / (count ** 0.5))
    cv = std_dev / mean if mean != 0 else 0
    confidence_percent = cv * 100  # Coefficient of Variation as a percentage
    return mean, std_dev, cv, confidence_percent, ci_lower, ci_upper

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

# Process team rosters
def process_team_roster(team_name):
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            normalized_positions = normalize_player_position(player_position)
            print(f"Processed {player_name} with positions {normalized_positions}")

    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Main analysis for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        return

    for game in games_today:
        print(f"Processing game: {game['home_team']} vs. {game['away_team']}")
        for team_name in [game['home_team'], game['away_team']]:
            process_team_roster(team_name)
    for pos in normalized_positions:
        dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics['TEAM_ABBREVIATION'], 1)
        # Proceed with calculations or aggregations

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Analyze teams and players for today's games
# Main analysis function
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    # Ensure games_today has been fetched
    games_today = fetch_schedule()

    if not games_today:
        print("No games scheduled for today.")
    else:
        all_players_data = []  # You need to define this to collect player data
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']

            for team_name in [home_team, away_team]:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    # Process player data as needed
                    # Your processing logic goes here

            for game in games_today:
                home_team = game['home_team']
                away_team = game['away_team']

                for team_name in [home_team, away_team]:
                    team_id = teams.find_team_by_abbreviation(team_name)['id']
                    roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                    for _, player in roster.iterrows():
                        player_position = player['POSITION']
                        # Continue processing

                    if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                    for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = calculate_stats_with_ci(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg, confidence_pct_rpg, ci_lower_rpg, ci_upper_rpg = calculate_stats_with_ci(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg, confidence_pct_apg, ci_lower_apg, ci_upper_apg = calculate_stats_with_ci(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                ppg_diff = mean_ppg - std_dev_ppg
                                rpg_diff = mean_rpg - std_dev_rpg
                                apg_diff = mean_apg - std_dev_apg

                                all_players_data.append({
                                    'Player': player_name,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CI Lower': round(ci_lower_ppg, 2),
                                    'PPG CI Upper': round(ci_upper_ppg, 2),
                                    'Confidence % PPG': round(confidence_pct_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': round(ppg_diff, 2),  # Added PPG Diff
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CI Lower': round(ci_lower_rpg, 2),
                                    'RPG CI Upper': round(ci_upper_rpg, 2),
                                    'Confidence % RPG': round(confidence_pct_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': round(rpg_diff, 2),  # Added RPG Diff
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CI Lower': round(ci_lower_apg, 2),
                                    'APG CI Upper': round(ci_upper_apg, 2),
                                    'Confidence % APG': round(confidence_pct_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': round(apg_diff, 2),  # Added APG Diff
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })
                        except Exception as e:
                                print(f"Error analyzing team {team_name}: {e}")
                                continue

    df = pd.DataFrame(all_players_data)
    display(df)
    return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()



Error fetching DvP stats: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)


ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Function to calculate statistics with confidence intervals
def calculate_stats_with_ci(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    count = len(stat_series)
    ci_lower = mean - 1.96 * (std_dev / (count ** 0.5))
    ci_upper = mean + 1.96 * (std_dev / (count ** 0.5))
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv, ci_lower, ci_upper

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

time.sleep(1)  # Delay 1 second between requests

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

time.sleep(1)  # Delay 1 second between requests

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

time.sleep(1)  # Delay 1 second between requests

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

time.sleep(1)  # Delay 1 second between requests

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

time.sleep(1)  # Delay 1 second between requests

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

time.sleep(1)  # Delay 1 second between requests

# Process team rosters
def process_team_roster(team_name):
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            normalized_positions = normalize_player_position(player_position)
            print(f"Processed {player_name} with positions {normalized_positions}")

    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

time.sleep(1)  # Delay 1 second between requests

# Main analysis for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    if not games_today:
        return

    for game in games_today:
        print(f"Processing game: {game['home_team']} vs. {game['away_team']}")
        for team_name in [game['home_team'], game['away_team']]:
            process_team_roster(team_name)
    for pos in normalized_positions:
        dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics['TEAM_ABBREVIATION'], 1)
        # Proceed with calculations or aggregations

time.sleep(1)  # Delay 1 second between requests

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

time.sleep(1)  # Delay 1 second between requests

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

time.sleep(1)  # Delay 1 second between requests

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

time.sleep(1)  # Delay 1 second between requests

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

time.sleep(1)  # Delay 1 second between requests

# Analyze teams and players for today's games
# Main analysis function
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    # Ensure games_today has been fetched
    games_today = fetch_schedule()

    if not games_today:
        print("No games scheduled for today.")
    else:
        all_players_data = []  # You need to define this to collect player data
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']

            for team_name in [home_team, away_team]:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    # Process player data as needed
                    # Your processing logic goes here

            for game in games_today:
                home_team = game['home_team']
                away_team = game['away_team']

                for team_name in [home_team, away_team]:
                    team_id = teams.find_team_by_abbreviation(team_name)['id']
                    roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                    for _, player in roster.iterrows():
                        player_position = player['POSITION']
                        # Continue processing

                    if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                    for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')
                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = calculate_stats_with_ci(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg, confidence_pct_rpg, ci_lower_rpg, ci_upper_rpg = calculate_stats_with_ci(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg, confidence_pct_apg, ci_lower_apg, ci_upper_apg = calculate_stats_with_ci(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                ppg_diff = mean_ppg - std_dev_ppg
                                rpg_diff = mean_rpg - std_dev_rpg
                                apg_diff = mean_apg - std_dev_apg

                                all_players_data.append({
                                    'Player': player_name,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CI Lower': round(ci_lower_ppg, 2),
                                    'PPG CI Upper': round(ci_upper_ppg, 2),
                                    'Confidence % PPG': round(confidence_pct_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': round(ppg_diff, 2),  # Added PPG Diff
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CI Lower': round(ci_lower_rpg, 2),
                                    'RPG CI Upper': round(ci_upper_rpg, 2),
                                    'Confidence % RPG': round(confidence_pct_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': round(rpg_diff, 2),  # Added RPG Diff
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CI Lower': round(ci_lower_apg, 2),
                                    'APG CI Upper': round(ci_upper_apg, 2),
                                    'Confidence % APG': round(confidence_pct_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': round(apg_diff, 2),  # Added APG Diff
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })
                        except Exception as e:
                                print(f"Error analyzing team {team_name}: {e}")
                                continue

    df = pd.DataFrame(all_players_data)
    display(df)
    return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()



Error analyzing team MIL: name 'confide' is not defined
Error analyzing team ORL: name 'confide' is not defined
Error analyzing team MIL: name 'confide' is not defined
Error analyzing team ORL: name 'confide' is not defined
Error analyzing team OKC: name 'confide' is not defined
Error analyzing team DAL: name 'confide' is not defined
Error analyzing team OKC: name 'confide' is not defined
Error analyzing team DAL: name 'confide' is not defined
Error analyzing team MIL: name 'confide' is not defined
Error analyzing team ORL: name 'confide' is not defined
Error analyzing team MIL: name 'confide' is not defined
Error analyzing team ORL: name 'confide' is not defined
Error analyzing team OKC: name 'confide' is not defined
Error analyzing team DAL: name 'confide' is not defined


ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog
from IPython.display import display

# Function to calculate statistics with confidence intervals
def calculate_stats_with_ci(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    count = len(stat_series)
    ci_lower = mean - 1.96 * (std_dev / (count ** 0.5))
    ci_upper = mean + 1.96 * (std_dev / (count ** 0.5))
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv, ci_lower, ci_upper

    def calculate_stats_with_ci(stat_series):
    # Ensure the series is not empty
     if len(stat_series) == 0:
        return 0, 0, 0, 0, 0, 0  # Default values if no data is available

    mean = stat_series.mean()
    std_dev = stat_series.std()
    count = len(stat_series)
    ci_lower = mean - 1.96 * (std_dev / (count ** 0.5))
    ci_upper = mean + 1.96 * (std_dev / (count ** 0.5))
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv, ci_lower, ci_upper

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Delay 1 second between requests to avoid overloading servers
time.sleep(1)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

time.sleep(1)

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

time.sleep(1)

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

time.sleep(1)

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25", season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

time.sleep(1)

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

time.sleep(1)

# Process team rosters
def process_team_roster(team_name):
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            normalized_positions = normalize_player_position(player_position)
            print(f"Processed {player_name} with positions {normalized_positions}")

    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

time.sleep(1)

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        logs = playergamelog.PlayerGameLog(player_id=player_id, season="2024-25")
        game_logs = logs.get_data_frames()[0]
        return game_logs
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error

# Main analysis for today's games
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
    else:
        all_players_data = []
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']

            for team_name in [home_team, away_team]:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                try:
                    roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]
                except KeyError:
                    print(f"Roster not found for {team_name}. Skipping...")
                    continue

                for _, player in roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    player_position = player['POSITION']
                    usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                    injury_status = injury_data.get(player_name, 'Healthy')
                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                        # Update the calculation to print the results for inspection
                        mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = (None, None, None, None, None, None)

                        try:
                            stats_ppg = calculate_stats_with_ci(game_logs['PTS'])
                            print("PPG Stats:", stats_ppg)  # Print the result for inspection
                            mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = stats_ppg
                        except Exception as e:
                             print(f"Error processing PPG stats for player {player_name}: {e}")

                        try:
                            stats_rpg = calculate_stats_with_ci(game_logs['REB'])
                            print("RPG Stats:", stats_rpg)  # Print the result for inspection
                            mean_rpg, std_dev_rpg, cv_rpg, confidence_pct_rpg, ci_lower_rpg, ci_upper_rpg = stats_rpg
                        except Exception as e:
                            print(f"Error processing RPG stats for player {player_name}: {e}")

                        try:
                            stats_apg = calculate_stats_with_ci(game_logs['AST'])
                            print("APG Stats:", stats_apg)  # Print the result for inspection
                            mean_apg, std_dev_apg, cv_apg, confidence_pct_apg, ci_lower_apg, ci_upper_apg = stats_apg
                        except Exception as e:
                            print(f"Error processing APG stats for player {player_name}: {e}")

                        # Handle adjusted predictions (make sure to check if necessary data exists)
                        adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                        adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                        adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                        # Add new difference columns
                        ppg_diff = mean_ppg - std_dev_ppg
                        rpg_diff = mean_rpg - std_dev_rpg
                        apg_diff = mean_apg - std_dev_apg

                        all_players_data.append({
                            'Player': player_name,
                            'Team': team_name,
                            'PPG': round(mean_ppg, 2),
                            'PPG SD': round(std_dev_ppg, 2),
                            'PPG CI Lower': round(ci_lower_ppg, 2),
                            'PPG CI Upper': round(ci_upper_ppg, 2),
                            'RPG': round(mean_rpg, 2),
                            'RPG SD': round(std_dev_rpg, 2),
                            'RPG CI Lower': round(ci_lower_rpg, 2),
                            'RPG CI Upper': round(ci_upper_rpg, 2),
                            'APG': round(mean_apg, 2),
                            'APG SD': round(std_dev_apg, 2),
                            'APG CI Lower': round(ci_lower_apg, 2),
                            'APG CI Upper': round(ci_upper_apg, 2),
                            'Adjusted PPG': round(adjusted_ppg, 2),
                            'Adjusted RPG': round(adjusted_rpg, 2),
                            'Adjusted APG': round(adjusted_apg, 2),
                            'Confidence PPG': confidence_pct_ppg,
                            'Confidence RPG': confidence_pct_rpg,
                            'Confidence APG': confidence_pct_apg,
                            'USG%': usg_pct,
                            'Injury Status': injury_status
                        })

                       except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                        continue

    df = pd.DataFrame(all_players_data)
    display(df)
return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 304)

In [ ]:
!pip install nba_api
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players

def validate_game_log(player_name, game_logs):
    """
    Validate that each game log contains exactly 6 columns.
    """
    required_columns = 6
    for idx, log in enumerate(game_logs):
        if len(log) != required_columns:
            print(f"Error for player {player_name}, game {idx + 1}: Expected {required_columns} columns, got {len(log)}")
            print(f"Data: {log}")
            return False
    print(f"All game logs for {player_name} are valid.")
    return True

# Fetch game logs for Damian Lillard
def fetch_and_validate_lillard():
    try:
        # Fetch Damian Lillard's ID
        player_name = "Damian Lillard"
        player_info = players.find_players_by_full_name(player_name)
        if not player_info:
            print(f"Player {player_name} not found.")
            return

        player_id = player_info[0]['id']

        # Fetch game logs
        game_log = playergamelog.PlayerGameLog(player_id=player_id, season='2023-24', season_type_all_star='Regular Season')
        game_log_data = game_log.get_data_frames()[0]

        # Select specific columns to match the expected data
        columns_to_check = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'TOV']
        if not all(col in game_log_data.columns for col in columns_to_check):
            print(f"Some required columns are missing in data for {player_name}.")
            return

        # Extract the relevant data as lists
        relevant_logs = game_log_data[columns_to_check].values.tolist()

        # Validate the logs
        validate_game_log(player_name, relevant_logs)

    except Exception as e:
        print(f"Error processing player {player_name}: {e}")

# Run the validation for Damian Lillard
fetch_and_validate_lillard()


All game logs for Damian Lillard are valid.


In [ ]:
# Example of a function to validate the number of columns for each player's game logs
def validate_player_data(player_name, game_logs):
    """
    Validate that each row in the game logs contains exactly 6 columns.
    """
    required_columns = 6  # Number of expected columns
    for idx, log in enumerate(game_logs):
        if len(log) != required_columns:
            print(f"Error for player {player_name}, game {idx + 1}: Expected {required_columns} columns, got {len(log)}")
            print(f"Data: {log}")
            return False
    print(f"All data for player {player_name} is valid.")
    return True

# Example usage
# Simulated game logs for testing (replace with actual API data)
players_game_logs = {
    "Damian Lillard": [
        [25, 5, 7, 1, 0, 3],  # Example of valid data
        [30, 4, 5, 2, 1],      # Invalid: Only 5 columns
    ],
    "Another Player": [
        [15, 10, 3, 0, 0, 2],  # Valid
        [20, 8, 2, 1, 1, 1],   # Valid
    ]
}

# Validate each player's data
for player, logs in players_game_logs.items():
    validate_player_data(player, logs)


Error for player Damian Lillard, game 2: Expected 6 columns, got 5
Data: [30, 4, 5, 2, 1]
All data for player Another Player is valid.


In [ ]:
!pip install nba_api
from nba_api.stats.endpoints import playergamelog
# Example headers
column_headers = ["PTS", "REB", "AST", "STL", "BLK", "TOV"]

# Example player data
players_game_logs = {
    "Damian Lillard": [
        [25, 5, 7, 1, 0, 3],  # Valid row
        [30, 4, 5, 2, 1],     # Invalid row: Missing one column
    ]
}

# Print column headers and validate data
print("Column Headers:", column_headers)

def validate_player_data(player_name, game_logs):
    """
    Validate that each row in the game logs contains exactly 6 columns.
    """
    required_columns = len(column_headers)  # Number of expected columns
    for idx, log in enumerate(game_logs):
        if len(log) != required_columns:
            print(f"Error for player {player_name}, game {idx + 1}: Expected {required_columns} columns, got {len(log)}")
            print(f"Data: {log}")
            return False
    print(f"All data for player {player_name} is valid.")
    return True

# Validate player data
for player, logs in players_game_logs.items():
    validate_player_data(player, logs)


Column Headers: ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV']
Error for player Damian Lillard, game 2: Expected 6 columns, got 5
Data: [30, 4, 5, 2, 1]


In [ ]:
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog
from IPython.display import display

# Function to calculate statistics with confidence intervals
def calculate_stats_with_ci(stat_series):
    # Ensure the series is not empty
    if len(stat_series) == 0:
        return 0, 0, 0, 0, 0  # Default values if no data is available

    mean = stat_series.mean()
    std_dev = stat_series.std()
    count = len(stat_series)
    ci_lower = mean - 1.96 * (std_dev / (count ** 0.5))
    ci_upper = mean + 1.96 * (std_dev / (count ** 0.5))
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv, ci_lower, ci_upper

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Delay 1 second between requests to avoid overloading servers
time.sleep(1)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

time.sleep(1)

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

time.sleep(1)

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

time.sleep(1)

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25", season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

time.sleep(1)

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

time.sleep(1)

# Process team rosters
def process_team_roster(team_name):
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            normalized_positions = normalize_player_position(player_position)
            print(f"Processed {player_name} with positions {normalized_positions}")

    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

time.sleep(1)

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        logs = playergamelog.PlayerGameLog(player_id=player_id, season="2024-25")
        game_logs = logs.get_data_frames()[0]
        return game_logs
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error

# Analyze teams for today
def analyze_teams_for_today():
    try:
        print("Initializing data...")
        dvp_data = initialize_dvp_data()
        print("Fetching today's schedule...")
        games_today = fetch_schedule()

        if not games_today:
            print("No games scheduled for today.")
            return

        team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
        player_usg_data = fetch_player_usg()
        injury_data = fetch_injury_data()

        print(f"Found {len(games_today)} games today.")
        all_players_data = []

        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']

            for team_name in [home_team, away_team]:
                print(f"Processing team: {team_name}")
                team_id = teams.find_team_by_abbreviation(team_name)['id']

                try:
                    print(f"Fetching roster for {team_name}...")
                    roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]
                except KeyError as e:
                    print(f"Error fetching roster for {team_name}: {e}. Skipping...")
                    continue

                for _, player in roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    player_position = player['POSITION']
                    usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                    injury_status = injury_data.get(player_name, 'Healthy')
                    print(f"Processing player: {player_name} (Position: {player_position})")

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        print(f"No game logs found for player {player_name}. Skipping...")
                        continue

                    # Initialize variables
                    mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = (None, None, None, None, None, None)

                    try:
                        stats_ppg = calculate_stats_with_ci(game_logs['PTS'])
                        mean_ppg, std_dev_ppg, cv_ppg, confidence_pct_ppg, ci_lower_ppg, ci_upper_ppg = stats_ppg
                        print(f"PPG Stats for {player_name}: {stats_ppg}")
                    except Exception as e:
                        print(f"Error processing PPG stats for player {player_name}: {e}")

                    try:
                        stats_rpg = calculate_stats_with_ci(game_logs['REB'])
                        mean_rpg, std_dev_rpg, cv_rpg, confidence_pct_rpg, ci_lower_rpg, ci_upper_rpg = stats_rpg
                        print(f"RPG Stats for {player_name}: {stats_rpg}")
                    except Exception as e:
                        print(f"Error processing RPG stats for player {player_name}: {e}")

                    try:
                        stats_apg = calculate_stats_with_ci(game_logs['AST'])
                        mean_apg, std_dev_apg, cv_apg, confidence_pct_apg, ci_lower_apg, ci_upper_apg = stats_apg
                        print(f"APG Stats for {player_name}: {stats_apg}")
                    except Exception as e:
                        print(f"Error processing APG stats for player {player_name}: {e}")

                    # Adjust predictions
                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], avg_pace, avg_off_rating, avg_def_rating)

                    # Append to list
                    all_players_data.append({
                        'Player': player_name,
                        'Team': team_name,
                        'PPG': round(mean_ppg, 2),
                        'PPG SD': round(std_dev_ppg, 2),
                        'PPG CI Lower': round(ci_lower_ppg, 2),
                        'PPG CI Upper': round(ci_upper_ppg, 2),
                        'RPG': round(mean_rpg, 2),
                        'RPG SD': round(std_dev_rpg, 2),
                        'RPG CI Lower': round(ci_lower_rpg, 2),
                        'RPG CI Upper': round(ci_upper_rpg, 2),
                        'APG': round(mean_apg, 2),
                        'APG SD': round(std_dev_apg, 2),
                        'APG CI Lower': round(ci_lower_apg, 2),
                        'APG CI Upper': round(ci_upper_apg, 2),
                        'Adjusted PPG': round(adjusted_ppg, 2),
                        'Adjusted RPG': round(adjusted_rpg, 2),
                        'Adjusted APG': round(adjusted_apg, 2),
                        'Confidence PPG': confidence_pct_ppg,
                        'Confidence RPG': confidence_pct_rpg,
                        'Confidence APG': confidence_pct_apg,
                        'USG%': usg_pct,
                        'Injury Status': injury_status
                    })

        # Convert to DataFrame
        player_df = pd.DataFrame(all_players_data)
        display(player_df)

    except Exception as e:
        print(f"Error analyzing teams for today: {e}")


from google.colab import data_table
data_table.disable_dataframe_formatter()